In [2]:
import numpy as np
from scipy.spatial.transform import Rotation
from math import cos, sin, pi
from h5_utils import load, save
from quat_utils import quatconj, quatmultiply
import os
os.environ.pop('DISPLAY', None)
import h5py
import pyvista as pv
import vtk
print(type(vtk.vtkRenderWindow()).__name__)   # confirm: vtkOSOpenGLRenderWindow
from scipy.linalg import eig
# from rpca import rpca_on_scans
from rpca_ortho import rpca_on_scans

vtkOSOpenGLRenderWindow


In [ ]:
# 1. Define the file path
FILE_PATH = '6_TRAJECTORY_AND_SONAR.h5'

# 2. Load the Trajectory (H)
TRAJ = load(FILE_PATH, 'TRAJ')
H    = TRAJ.H       # Shape: (N, 7) - [x, y, z, qw, qx, qy, qz]
TIME = TRAJ.TIME    # Timestamps for the poses

# 3. Load Vertical Sonar Data (M^v)
M_v_data = load(FILE_PATH, 'M_v')
rho_v      = M_v_data.rho       # Distance/Range measurements
psi_v      = M_v_data.psi       # Rotation angles
pose_idx_v = M_v_data.pose_idx.astype(int)  # The mapping function \sigma^v(i)

# 4. Load Horizontal Sonar Data (M^h)
M_h_data = load(FILE_PATH, 'M_h')
# M_h_data.M is a (3, total_returns) array where:
# Row 0 = rho_h, Row 1 = theta_h (NaNs to be estimated), Row 2 = psi_h
rho_h      = M_h_data.M[0, :]   
theta_h    = M_h_data.M[1, :]   # Currently NaNs
psi_h      = M_h_data.M[2, :]   
pose_idx_h = M_h_data.pose_idx.astype(int)  # The mapping function \sigma^h(i)

In [ ]:
# ---------------------------------------------------------------------------
# Helpers: SE(3) ↔ matrix conversions
# ---------------------------------------------------------------------------

def pose_to_SE3(pose):
    """
    Convert a 7-vector pose [x, y, z, qw, qx, qy, qz] to a 4×4 SE(3) matrix.

    The notebook stores poses as:
        H = np.hstack((X_IMU_ARRAY, QUAT_IMU_ARRAY))   # shape (N, 7)
        where QUAT_IMU is [qw, qx, qy, qz]  (scalar-first convention).
    """
    t              = pose[:3]          # translation [x, y, z]
    qw, qx, qy, qz = pose[3:]          # quaternion (scalar first)
    R              = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()   # 3×3 # scipy.spatial.transform.Rotation uses scalar-last [qx, qy, qz, qw]

    T         = np.eye(4)
    T[:3, :3] = R
    T[:3,  3] = t
    return T

def SE3_log(T):
    """
    SE(3) logarithm map.

    Given a 4×4 transformation matrix T = [R | t ; 0 | 1],
    returns the 6-vector twist  ξ = [v ; ω]  in  se(3).

    The rotation part ω is the axis-angle vector (‖ω‖ = rotation angle).
    The translation part v satisfies  t = V ω  where V is the left-Jacobian
    of SO(3).

    Reference: Eq. (13) in the paper, plus standard SE(3) Lie theory.
    """
    R = T[:3, :3] # Rotation part
    t = T[:3,  3] # Translation part

    # --- rotation part: ω (axis-angle) ---
    omega = Rotation.from_matrix(R).as_rotvec()   # 3-vector; ‖ω‖ = angle θ
    theta = np.linalg.norm(omega)

    # --- translation part: v = V⁻¹ t ---
    if theta < 1e-9:
        # First-order approximation when rotation is negligible
        v = t.copy()
    else:
        # Skew-symmetric matrix of ω (not unit vector)
        W = np.array([
            [ 0,        -omega[2],  omega[1]],
            [ omega[2],  0,        -omega[0]],
            [-omega[1],  omega[0],  0       ]
        ])
        # Left Jacobian of SO(3):
        #   V = I  +  (1 - cosθ)/θ²  ·  W  +  (θ - sinθ)/θ³  ·  W²
        V = (np.eye(3)
             + (1 - np.cos(theta)) / theta**2 * W
             + (theta - np.sin(theta)) / theta**3 * (W @ W))

        v = np.linalg.solve(V, t)   # v = V⁻¹ t

    return np.concatenate([v, omega])   # 6-vector [v; ω]

# ---------------------------------------------------------------------------
# Algorithm 1 — Line 1:  s = s(H)
# ---------------------------------------------------------------------------

def compute_curvilinear_abscissa(H):
    """
    Compute the curvilinear abscissa for every pose in the trajectory.

    Implements Equation 12  s(t) = L_{η(0)}^{η(t)}
    via the discrete approximation of Equation 13:

        s_i = Σ_{k=1}^{i}  sqrt( log(T_{k-1}⁻¹ T_k)ᵀ G log(T_{k-1}⁻¹ T_k) )

    where the 6×6 matrix G = diag(I₃, 2·I₃)  (Eq. 10) weights translations
    by 1 and rotations by 2, yielding the norm:

        ‖ξ‖_G  =  sqrt( ‖v‖² + 2‖ω‖² )

    Parameters
    ----------
    H : np.ndarray, shape (N, 7)
        Trajectory array.  Each row is [x, y, z, qw, qx, qy, qz].

    Returns
    -------
    s : np.ndarray, shape (N,)
        Cumulative arc length (curvilinear abscissa) for each pose.
        s[0] = 0 by definition (Eq. 12, length from the start).
    """
    N = H.shape[0]
    s = np.zeros(N)   # s[0] = 0 by definition

    # G is block-diagonal: translation weight = 1, rotation weight = 2
    # ‖ξ‖_G² = ξᵀ G ξ = ‖v‖² + 2‖ω‖²
    # We compute this directly rather than building the 6×6 matrix.

    for i in range(1, N):
        T_prev   = pose_to_SE3(H[i - 1])   # T_{k-1}
        T_curr   = pose_to_SE3(H[i])       # T_k

        # Relative (incremental) transformation: ΔT = T_{k-1}⁻¹ · T_k
        # For SE(3): inverse is  [Rᵀ | -Rᵀt ; 0 | 1]
        T_rel    = np.linalg.inv(T_prev) @ T_curr
        xi       = SE3_log(T_rel)  # SE(3) log map → 6-vector [v ; ω]
        v, omega = xi[:3], xi[3:]
        delta_s  = np.sqrt(np.dot(v, v) + 2.0 * np.dot(omega, omega)) # Weighted norm  sqrt(‖v‖² + 2‖ω‖²)  — direct expansion of ξᵀ G ξ
        s[i]     = s[i - 1] + delta_s # Accumulate  (Eq. 13)

    return s

H is "where the robot is," and s is "how far the robot has traveled along its path."

# 1)    $\mathbf{s}=\begin{bmatrix}s_1&\dots&s_{n_r}\end{bmatrix}^T\leftarrow s(H)$ -- curvilinear abscissa for the whole trajectory



In [ ]:

s_traj = compute_curvilinear_abscissa(H)

# Horizontal Sonar Full-Scan Detection & Data Structuring

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h3 style="color: #5d4e37; text-align: center; margin-bottom: 20px; font-weight: bold; max-width: 100%; word-wrap: break-word; hyphens: auto;">Horizontal Sonar Full-Scan Detection & Data Structuring</h3>

<p style="text-align: justify; margin: 20px 0; color: #4a4a4a; width: 100%; max-width: 100%; word-wrap: break-word; overflow-wrap: break-word; hyphens: auto; box-sizing: border-box;">
This section groups the continuous stream of individual sonar pings into <strong>complete 360° rotational scans</strong>. By analyzing the angle of the horizontal Ping360 sonar, the algorithm detects when the beam angle resets (wraps around), marking the boundary of a full physical sweep.
</p>

<div style="background-color: #f4eee1; padding: 15px; border-left: 4px solid #bda881; border-radius: 4px; margin: 20px 0;">
<h4 style="margin-top: 0; color: #5d4e37; font-size: 0.95em; font-weight: bold;">The resulting structure, <code style="background-color: #e6ddc5; padding: 2px 5px; border-radius: 3px;">SCAN_DATA</code>, groups the data into 5 key columns per scan:</h4>
<ol style="margin: 10px 0 0 0; color: #4a4a4a; font-size: 0.9em; padding-left: 20px;">
  <li style="margin-bottom: 8px;"><strong>Scan Cycle:</strong> The sequential index of the full 360° scan.</li>
  <li style="margin-bottom: 8px;"><strong>M<sub>v</sub> (Vertical Sonar):</strong> The time-filtered impact points from the SeaKing profiling sonar that fired <em>during</em> this specific scan's time window.</li>
  <li style="margin-bottom: 8px;"><strong>M<sub>h</sub> (Horizontal Sonar):</strong> The valid impact points from the Ping360 sonar across all angles in this rotation.</li>
  <li style="margin-bottom: 8px;"><strong>H (Trajectory):</strong> The exact kinematic poses <code style="background-color: #e6ddc5; padding: 2px 4px; border-radius: 3px;">[x, y, z, qw, qx, qy, qz]</code> the robot executed while performing this sweep.</li>
  <li style="margin-bottom: 0;"><strong>s (Curvilinear Abscissa):</strong> The exact distance the robot traveled along its path during this sweep.</li>
</ol>
</div>

<p style="text-align: justify; margin: 0; color: #4a4a4a; width: 100%; max-width: 100%; word-wrap: break-word; font-style: italic; box-sizing: border-box;">
This guarantees that for any given 360° sonar image, we have the corresponding 3D timeline of exactly where the robot was matching that exact time frame.
</p>

</div>

In [ ]:
PING360_VALID = load('3_VALID_DATA.h5', 'PROFILO_PING360_VALID')
POSE_IDX = load('6_TRAJECTORY_AND_SONAR.h5', 'POSE_IDX')
PING360_POSE_IDX = POSE_IDX.PING360_POSE_IDX.astype(int)

# ---------------------------------------------------------------------------
# Detect full-scan boundaries of the horizontal sonar (Ping360)
# ---------------------------------------------------------------------------
# A new scan starts whenever the angle decreases (wraps around)
ping360_angles = PING360_VALID.ANGLE  # already loaded from Input.ipynb pipeline # already loaded from Input.ipynb pipeline

scan_starts = [0]  # First ping always starts a scan
for i in range(1, len(ping360_angles)):
    if ping360_angles[i] < ping360_angles[i - 1]:  # angle wrapped → new scan
        scan_starts.append(i)
scan_starts.append(len(ping360_angles))  # sentinel for the last scan

n_scans = len(scan_starts) - 1
print(f"Detected {n_scans} full scans")

# ---------------------------------------------------------------------------
# Build the per-scan data structure
# ---------------------------------------------------------------------------
SCAN_DATA = []

for k in range(n_scans):
    i_start = scan_starts[k]
    i_end   = scan_starts[k + 1]   # exclusive

    # 1) Scan number (1-indexed)
    scan_number = k + 1

    # 2) Collect M_v entries whose time falls within this scan's time window
    t_scan_start = PING360_VALID.ABS_TIME[i_start]
    t_scan_end   = PING360_VALID.ABS_TIME[i_end - 1]

    # M_v entries in this time window (use M_v_data arrays)
    mv_mask = (M_v_data.time >= t_scan_start) & (M_v_data.time <= t_scan_end)
    mv_in_scan = {
        'rho':      M_v_data.rho[mv_mask],
        'psi':      M_v_data.psi[mv_mask],
        'pose_idx': M_v_data.pose_idx[mv_mask].astype(int),
        'time':     M_v_data.time[mv_mask]
    }
    # M_h entries in this time window (use M_h_data arrays)
    mh_mask = (M_h_data.time >= t_scan_start) & (M_h_data.time <= t_scan_end)
    mh_in_scan = {
        'M':        M_h_data.M[:, mh_mask],
        'pose_idx': M_h_data.pose_idx[mh_mask].astype(int),
        'time':     M_h_data.time[mh_mask]
}
    # (or filter by time similarly)

    # 4) Trajectory H during this scan:
    #    All pose indices referenced by pings in this scan
    pose_indices = PING360_POSE_IDX[i_start:i_end]
    idx_min = min(pose_indices)
    idx_max = max(pose_indices)
    H_scan = H[idx_min:idx_max + 1, :]   # shape (n_poses_in_scan, 7)

    # 5) Curvilinear abscissa for those poses
    s_scan = s_traj[idx_min:idx_max + 1]  # shape (n_poses_in_scan,)

    scan_entry = {
        'scan_number': scan_number,         # Column 1: scan ID
        'M_v':         mv_in_scan,          # Column 2: vertical sonar impacts
        'M_h':         mh_in_scan,          # Column 3: horizontal sonar impacts
        'H':           H_scan,              # Column 4: trajectory during this scan
        's':           s_scan,              # Column 5: curvilinear abscissa
        'pose_range':  (idx_min, idx_max),  # bonus: which poses this covers
    }
    SCAN_DATA.append(scan_entry)

print(f"Built {len(SCAN_DATA)} scan entries")
print(f"Example — Scan 1: {SCAN_DATA[0]['H'].shape[0]} poses, "
      f"s from {SCAN_DATA[0]['s'][0]:.2f} to {SCAN_DATA[0]['s'][-1]:.2f}")


<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 15px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h4 style="color: #5d4e37; text-align: center; margin-bottom: 15px; font-weight: bold;">3D Point Cloud from Vertical Sonar — Per-Scan Generation</h4>

<p style="text-align: justify; margin: 10px 0; color: #4a4a4a;">
For each scan, the vertical sonar measurements (M<sub>v</sub>) are transformed from the sensor's body-frame into world coordinates using the robot's trajectory. The quaternion-based coordinate transformation pipeline is:
</p>

<ol style="margin: 10px 0; color: #4a4a4a; padding-left: 20px;">
  <li>Convert sonar angle ψ → body-frame quaternion Q<sub>body</sub></li>
  <li>Apply frame alignment rotations (Z by π/2, Y by π/2)</li>
  <li>Combine: Q<sub>combined</sub> = Q<sub>robot</sub> ⊗ Q<sub>body</sub></li>
  <li>Rotate distance: V = Q<sub>combined</sub> ⊗ [0,0,0,ρ] ⊗ Q<sub>combined</sub>*</li>
  <li>World point: P<sub>world</sub> = P<sub>robot</sub> + V[1:4]</li>
  <li>Filter: keep Z &gt; 0</li>
</ol>

</div>

In [ ]:
def compute_point_cloud(M_v_rho, M_v_psi, M_v_pose_idx, H, pose_idx_min):
    """
    Generate a 3D point cloud from vertical sonar measurements.
    
    Parameters
    ----------
    M_v_rho       : (n,) distances from the sonar
    M_v_psi       : (n,) sonar rotation angles in degrees
    M_v_pose_idx  : (n,) global pose indices
    H             : (m, 7) local trajectory [x, y, z, qw, qx, qy, qz]
    pose_idx_min  : int, offset to convert global idx → local idx
    
    Returns
    -------
    point_cloud : (k, 3) array of [x, y, z] world-frame points (Z > 0 only)
    """
    points = []
    
    for i in range(len(M_v_rho)):
        # --- Look up robot pose ---
        local_idx = int(M_v_pose_idx[i] - pose_idx_min)
        
        # Safety clamp (in case of boundary issues)
        local_idx = max(0, min(local_idx, H.shape[0] - 1)) 
        
        CURRENT_X    = H[local_idx, :3]   # robot position [x, y, z]
        CURRENT_QUAT = H[local_idx, 3:]   # robot orientation [qw, qx, qy, qz]
        
        # --- Body-frame quaternion from sonar angle ---
        theta2rad =  M_v_psi[i] / 2
        CURRENT_IMPACT_QUAT_B = [cos(theta2rad), -sin(theta2rad), 0, 0]
        
        # Frame alignment rotations (same as 7_POINT_CLOUD.ipynb)
        Rot_Z_pi = [cos(pi / 2), 0, 0, sin(pi / 2)]
        CURRENT_IMPACT_QUAT_B = quatmultiply(CURRENT_IMPACT_QUAT_B, Rot_Z_pi)
        
        Rot_Y_pi = [cos(pi / 2), 0, sin(pi / 2), 0]
        CURRENT_IMPACT_QUAT_B = quatmultiply(CURRENT_IMPACT_QUAT_B, Rot_Y_pi)
        
        # --- Combined rotation: robot × body ---
        CURRENT_IMPACT_QUAT_0 = quatmultiply(CURRENT_QUAT, CURRENT_IMPACT_QUAT_B)
        
        # --- Rotate the distance vector ---
        Impact_Dist = np.array([0, 0, 0, M_v_rho[i]])
        Q_temp = quatmultiply(
            CURRENT_IMPACT_QUAT_0,
            quatmultiply(Impact_Dist, quatconj(CURRENT_IMPACT_QUAT_0))
        )
        
        # --- World-frame point ---
        IMPACT_TEMP = CURRENT_X + Q_temp[1:4]
        
        # Filter: keep only points with Z > 0
        if IMPACT_TEMP[2] > 0:
            points.append(IMPACT_TEMP)
    
    if len(points) > 0:
        return np.array(points)
    else:
        return np.empty((0, 3))

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 15px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h4 style="color: #5d4e37; text-align: center; margin-bottom: 15px; font-weight: bold;">Compute Point Clouds & Save All Scan Data</h4>

<p style="text-align: justify; margin: 10px 0; color: #4a4a4a;">
For each scan in <code>SCAN_DATA</code>, compute the 3D point cloud from the vertical sonar data, then flatten everything (including the <code>POINT_CLOUD</code>) into a single dict and save to <code>HDF5_DATA/7_SCANS.h5</code>.
</p>

<p style="text-align: justify; margin: 10px 0; color: #4a4a4a;">
The resulting HDF5 structure has scan groups <code>SCANS/0001</code>, <code>SCANS/0002</code>, ..., each containing: <code>M_v_rho</code>, <code>M_v_psi</code>, <code>M_v_pose_idx</code>, <code>M_v_time</code>, <code>M_h_M</code>, <code>M_h_pose_idx</code>, <code>M_h_time</code>, <code>H</code>, <code>s</code>, <code>scan_number</code>, <code>pose_idx_min</code>, <code>pose_idx_max</code>, and <code>POINT_CLOUD</code> (shape <code>(n_points, 3)</code>).
</p>

</div>

In [ ]:
# ---------------------------------------------------------------------------
# Compute point clouds for each scan and build the flattened save structure
# ---------------------------------------------------------------------------
FLATTENED_SCAN_DATA = []
scan_point_clouds = {}   # scan_number → point cloud array (for later export)
total_points = 0

print(f"Processing {len(SCAN_DATA)} scans...\n")

for idx, scan in enumerate(SCAN_DATA):
    # --- Compute the point cloud for this scan ---
    pc = compute_point_cloud(
        M_v_rho      = scan['M_v']['rho'],
        M_v_psi      = scan['M_v']['psi'],
        M_v_pose_idx = scan['M_v']['pose_idx'],
        H            = scan['H'],
        pose_idx_min = scan['pose_range'][0]
    )
    scan_point_clouds[scan['scan_number']] = pc
    total_points += pc.shape[0]
    
    # Progress update every 50 scans
    if (idx + 1) % 50 == 0 or (idx + 1) == len(SCAN_DATA):
        print(f"  Processed {idx + 1}/{len(SCAN_DATA)} scans  |  Scan {scan['scan_number']:04d}: {pc.shape[0]} points")

    # --- Build the flattened dict for saving ---
    flat_scan = {
        # WRAP SCALARS IN BRACKETS TO MAKE THEM 1D ARRAYS!
        'scan_number':  [scan['scan_number']],
        
        # Flatten M_v
        'M_v_rho':      scan['M_v']['rho'],
        'M_v_psi':      scan['M_v']['psi'],
        'M_v_pose_idx': scan['M_v']['pose_idx'],
        'M_v_time':     scan['M_v']['time'],
        
        # Flatten M_h
        'M_h_M':        scan['M_h']['M'],
        'M_h_pose_idx': scan['M_h']['pose_idx'],
        'M_h_time':     scan['M_h']['time'],
        
        # Direct Arrays
        'H':            scan['H'],
        's':            scan['s'],
        
        # WRAP SCALARS IN BRACKETS!
        'pose_idx_min': [scan['pose_range'][0]],
        'pose_idx_max': [scan['pose_range'][1]],
        
        # Point cloud (n_points, 3) — integrated from 7_point_cloud.ipynb
        'POINT_CLOUD':  pc,
    }
    FLATTENED_SCAN_DATA.append(flat_scan)

print(f"\n✓ Done! Total valid points across all scans: {total_points}")

# --- Save everything to HDF5 ---
OUTPUT_FILE = 'HDF5_DATA/7_SCANS.h5'
save(OUTPUT_FILE, 'SCANS', FLATTENED_SCAN_DATA, mode='w')

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 15px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h4 style="color: #5d4e37; text-align: center; margin-bottom: 15px; font-weight: bold;">Export Combined Point Cloud (PLY & PCD)</h4>

<p style="text-align: justify; margin: 10px 0; color: #4a4a4a;">
Combine all scan point clouds into one and export as <code>.ply</code> (via PyVista) and <code>.pcd</code> (via Open3D) for external visualization.
</p>

</div>

In [ ]:
# Combine all point clouds
all_points = np.vstack([pc for pc in scan_point_clouds.values() if pc.shape[0] > 0])
print(f"Total points: {all_points.shape[0]}")

output_dir = "Point_cloud"
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------------
# 1. Save as PLY using PyVista (includes elevation scalar)
# ---------------------------------------------------------
point_cloud = pv.PolyData(all_points)
point_cloud["elevation"] = all_points[:, 2]

ply_filename = os.path.join(output_dir, "fontaine_de_nimes_from_scans.ply")
point_cloud.save(ply_filename)
print(f"✓ PLY saved to: {ply_filename}")

# ---------------------------------------------------------
# 2. Save as PCD using Open3D
# ---------------------------------------------------------
try:
    import open3d as o3d
    pcd_filename = os.path.join(output_dir, "fontaine_de_nimes_from_scans.pcd")
    
    # Initialize Open3D point cloud object and assign points
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(all_points)
    
    # Save the file
    o3d.io.write_point_cloud(pcd_filename, pcd)
    print(f"✓ PCD saved to: {pcd_filename}")
    
except ImportError:
    print("\n⚠️ Note: Could not save .pcd. The 'open3d' library is required to save .pcd files.")
    print("You can install it by running: !pip install open3d in a new notebook cell.")

print(f"\nNumber of points processed: {len(all_points)}")

<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 15px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h4 style="color: #5d4e37; text-align: center; margin-bottom: 15px; font-weight: bold;">Quick Summary</h4>

<p style="text-align: justify; margin: 10px 0; color: #4a4a4a;">
Print a summary table showing the number of vertical sonar impacts vs. point cloud points per scan.
</p>

</div>

In [ ]:
print(f"{'Scan':>6}  {'M_v impacts':>12}  {'Point Cloud':>12}")
print("-" * 34)

for scan, pc in list(zip(SCAN_DATA, scan_point_clouds.values()))[:10]:
    key = f"{scan['scan_number']:04d}"
    n_mv = len(scan['M_v']['rho'])
    n_pc = pc.shape[0]
    print(f"{key:>6}  {n_mv:>12}  {n_pc:>12}")

print(f"{'...':>6}")

# Last scan
last_scan = SCAN_DATA[-1]
last_pc = list(scan_point_clouds.values())[-1]
last_key = f"{last_scan['scan_number']:04d}"
print(f"{last_key:>6}  {len(last_scan['M_v']['rho']):>12}  {last_pc.shape[0]:>12}")

In [ ]:
with h5py.File(OUTPUT_FILE, 'a') as f:
    for key, pc in scan_point_clouds.items():
        grp = f[f'SCANS/{key:04d}'] 
        
        # Remove old dataset if it exists
        if 'POINT_CLOUD' in grp:
            del grp['POINT_CLOUD']
        
        # Save with gzip compression
        grp.create_dataset('POINT_CLOUD', data=pc, compression='gzip', compression_opts=4)

print(f"✓ Saved POINT_CLOUD to all {len(scan_point_clouds)} scan groups in {OUTPUT_FILE}")

# 2) $f_p \leftarrow fitEllipticCylinder(\mathcal{M}^v)$ -- Surface prior



### 1 -- First we need a good initial shape. It is obtained by computing robust principal component analysis (RPCA) [32] on the vertical sonar point cloud. It provides three principals vectors.

In [ ]:
# ---------------------------------------------------------------------------
# Run Trimmed Grassmann Average (TGA) RPCA
# --------------------------------------------------------------------------

rpca_results = rpca_on_scans( # This will process all scans in the file, extracting K=3 components per scan.
    scan_ids = 'all',
    h5_path  = 'HDF5_DATA/7_SCANS.h5', # It automatically skips any scans with fewer than 3 points.
    K        = 3,
    trim_pct = 0.50,
    verbose  = True                    # Set to True to print progress and summaries for each scan.
)

In [ ]:
os.environ.pop('DISPLAY', None)   # clean up the zombie DISPLAY=:99 from earlier

from pyvista.trame.jupyter import launch_server
await launch_server().ready
                                                                     
from viz_utils import visualize_rpca_vectors
visualize_rpca_vectors(30, rpca_results)

Now we have three principal vectors.

### 2 -- To find the cylinder axis among those principal vectors, we used the fact that every tangent on a cylinder surface is orthogonal to its axis. This means that by approximating tangent vectors by the vector formed by two successive points, their projections on the PCA bases should be minimal along the cylinder axis.

In [ ]:
from cylinder_axis import find_cylinder_axes_on_scans

cylinder_axis = find_cylinder_axes_on_scans(scan_ids='all')

In [ ]:
from viz_utils import visualize_axis_selection

scan_num = 74
pc = cylinder_axis[scan_num]['point_cloud']
visualize_axis_selection(pc, cylinder_axis[scan_num], cylinder_axis[scan_num]['axis_index'])

One vector was identified as the cylinder axis (the length of the pipe)
The other two vectors span the plane of the cylinder’s cross-section. (cuts perpendicularly through the cylinder)

### 3 -- Then, the elliptic section is estimated by fitting the projections of all the points in the plane spanned by the two other principals axis [33]. The PCA base is then rotated around the cylinder axis to be aligned with the ellipse axis.

You have successfully projected your point cloud onto the plane spanned by Vectors 2 and 3, and you have fit a 2D ellipse to those points using the least-squares method from paper [33]. They are simply the directions of greatest and least variance in the data as computed by the PCA algorithm.

The Alignment Problem

While Vectors 2 and 3 perfectly define the plane of the ellipse, their actual pointing directions (the "PCA base") are completely arbitrary relative to the shape of the ellipse you just fit. They likely do not point straight down the longest part of the ellipse (the major axis) and the narrowest part (the minor axis).

The Solution (Rotating the Base)

To properly define your elliptic cylinder in 3D space as a structured pose, your local coordinate frame needs to be perfectly locked to the physical shape of the geometry.

    The Pivot: You treat Vector 1 (the cylinder axis) as a fixed hinge.

    The Rotation: You take the other two vectors (the PCA base) and rotate them together around that hinge.

    The Alignment: You stop rotating when one of the vectors points exactly along the major axis of your fitted 2D ellipse, and the other vector points exactly along the minor axis.

    deepseek:
    You spin it by just the right amount so that the frame's axes become perfectly aligned with the major and minor axes of the fitted ellipse. This is a 2D rotation in the plane orthogonal to the cylinder axis.

    cleude:
    So the rotation step is this: once you have the ellipse fit, you extract the orientation of the ellipse (i.e., the angle of its major axis relative to your current 2D basis in that plane), and you rotate the two PCA basis vectors — within the plane orthogonal to the cylinder axis — so that one basis vector aligns with the major ellipse axis and the other with the minor axis.

The Final Result

Once rotated, your three vectors now form a meaningful 3D basis (a 6D pose in SE(3)):

    Z-axis: Aligned with the length of the cylinder.

    X-axis: Aligned with the major axis of the elliptic cross-section.

    Y-axis: Aligned with the minor axis of the elliptic cross-section.

This perfectly aligned frame—along with the scale values of the major and minor axes—gives you the highly accurate "initial shape" required to kick off the Levenberg-Marquardt (LM) non-linear optimization step that follows.

In [ ]:
from fit_ellipse import fit_elliptic_sections

ellipse_results = fit_elliptic_sections(cylinder_axis) # Run the ellipse fitting function

# -------------------------------- Print ------------------------------------ #

if not ellipse_results:
    print("No valid ellipses could be fitted (points might be degenerate).")
else:
    for scan_num, ell in ellipse_results.items():
        print(f"--- Ellipse Fit Results for Scan {scan_num:04d} ---")
        print(f"  Center (2D PCA plane) : ({ell['center_2d'][0]:+.4f}, {ell['center_2d'][1]:+.4f})")
        print(f"  Semi-axes             : major={ell['semi_axes'][0]:.4f}, minor={ell['semi_axes'][1]:.4f}")
        print(f"  Orientation Angle     : {np.degrees(ell['angle_rad']):.2f}°")
        print(f"  Rotated Major Axis    : {ell['new_major']}")
        print(f"  Rotated Minor Axis    : {ell['new_minor']}")
        print("-" * 45)

Center (2D PCA plane) :

    Remember that you projected your 3D points onto a flat 2D plane (spanned by your two PCA vectors). On this plane, the origin (0,0) is your original point cloud's robust centroid. This result tells you that the true center of the fitted ellipse is offset by +0.7617 units along the first PCA axis and +1.7762 units along the second PCA axis from that centroid.

Semi-axes :

    The major value (...) is the semi-major axis, meaning it's the longest radius from the center to the edge of the ellipse. The minor value (...) is the semi-minor axis, or the shortest radius. Because they are not equal, it confirms your cross-section is distinctly elliptical, not a perfect circle.

Orientation Angle :

    Within your 2D projection plane, the ellipse is tilted. Specifically, the ellipse's major axis is rotated −28.27∘ (likely clockwise, depending on your axis orientation) relative to your original first PCA basis vector.

Rotated Major Axis :

    Your algorithm took the original 3D PCA vector and rotated it around the cylinder axis by that −28.27∘ angle. This vector now points exactly along the longest physical dimension of the ellipse in your absolute, global 3D world space.

Rotated Minor Axis :

    This points exactly along the shortest physical dimension of the ellipse in global 3D space. Mathematically, it is perpendicular to both the Rotated Major Axis and the original cylinder axis.

### 4 -- To further improve our estimation, we use the Levendberg–Marquardt (LM) algorithm to obtain the shape which minimizes orthogonal distances to the elliptic cylinder. The estimation vector $x \in \mathbb{R}^8$ representing our elliptic cylinder is composed of a 6D pose in $SE(3)$ and the scales along the minor and major ellipse axis. Here, we use on-manifold optimization on $SE(3)$, as described in [22].

In [16]:
# # Run on all scans
from lm_3 import optimize_all_scans, build_initial_state
opt_results = optimize_all_scans(cylinder_axis, ellipse_results, verbose=True)

  outer iter   1  |  ‖ε⋆‖ = 9.90e-01  |  a = 1.3067  b = 1.2105  |  RMS = 3.019583
  outer iter   2  |  ‖ε⋆‖ = 5.86e-01  |  a = 0.8932  b = 1.0409  |  RMS = 2.592771
  outer iter   3  |  ‖ε⋆‖ = 3.78e-07  |  a = 0.8932  b = 1.0409  |  RMS = 2.581569
  outer iter   4  |  ‖ε⋆‖ = 1.50e-16  |  a = 0.8932  b = 1.0409  |  RMS = 2.581525
  Converged: ‖ε⋆‖ = 1.50e-16 < tol = 1.0e-08

Optimization CONVERGED after 4 outer iterations
Total function evaluations: 122
Final semi-axes: a=0.8932, b=1.0409
RMS error: 2.581525

Processing Scan 0104
Starting on-manifold LM optimization with 204 points
Initial semi-axes: a=2.3883, b=2.1213
  outer iter   1  |  ‖ε⋆‖ = 1.05e+00  |  a = 1.4486  b = 1.2866  |  RMS = 3.310997
  outer iter   2  |  ‖ε⋆‖ = 8.01e-01  |  a = 0.9420  b = 0.8096  |  RMS = 2.578703
  outer iter   3  |  ‖ε⋆‖ = 3.44e-09  |  a = 0.9420  b = 0.8096  |  RMS = 2.578537
  Converged: ‖ε⋆‖ = 3.44e-09 < tol = 1.0e-08

Optimization CONVERGED after 3 outer iterations
Total function evaluations: 93

In [ ]:
import h5py

HDF5_FILE = 'HDF5_DATA/7_SCANS.h5'

print(f"\nSaving Surface Priors to {HDF5_FILE}...")

# 3. Open the HDF5 file in append mode ('a') so we don't overwrite existing data
with h5py.File(HDF5_FILE, 'a') as f:
    
    saved_count = 0
    for scan_num, opt_res in opt_results.items():
        scan_key = f"{scan_num:04d}"
        
        # Check if the scan exists in the HDF5 file
        if f"SCANS/{scan_key}" not in f:
            print(f"  ⚠ Warning: Scan {scan_key} not found in HDF5. Skipping.")
            continue
            
        grp = f[f"SCANS/{scan_key}"]
        
        # Create a clean subgroup for our Surface Prior data
        if 'SURFACE_PRIOR' in grp:
            del grp['SURFACE_PRIOR']  # Delete old one if re-running
            
        prior_grp = grp.create_group('SURFACE_PRIOR')
        
        # -------------------------------------------------------------
        # A. Save the INITIAL Shape (from Step 3 / PCA & Ellipse Fit)
        # -------------------------------------------------------------
        # We use the helper function from lm_3.py to build the initial T, a, b
        initial_state = build_initial_state(
            cylinder_axis[scan_num], 
            ellipse_results[scan_num]
        )
        
        init_grp = prior_grp.create_group('INITIAL')
        init_grp.create_dataset('T', data=initial_state['T'])
        init_grp.create_dataset('a', data=initial_state['a'])
        init_grp.create_dataset('b', data=initial_state['b'])
        
        # -------------------------------------------------------------
        # B. Save the OPTIMIZED Shape (from Step 4 / LM On-Manifold)
        # -------------------------------------------------------------
        opt_grp = prior_grp.create_group('OPTIMIZED')
        opt_grp.create_dataset('T', data=opt_res['T'])
        opt_grp.create_dataset('a', data=opt_res['a'])
        opt_grp.create_dataset('b', data=opt_res['b'])
        
        # Save a few extra diagnostic metrics just in case you need to 
        # filter out bad fits during the GP training phase
        opt_grp.create_dataset('rms_error', data=opt_res['rms_error'])
        opt_grp.create_dataset('success', data=opt_res['success'])
        
        saved_count += 1

print(f"✓ Successfully saved Surface Priors for {saved_count} scans!")

## 5 -- Visualize


In [ ]:
def _make_elliptic_cylinder_mesh(T, a, b, point_cloud):
    """Helper: Build a PyVista mesh for an elliptic cylinder from its parameters."""
    center = T[:3, 3]
    z_axis = T[:3, 2]

    # Project points onto the cylinder axis to find height bounds
    projections = (point_cloud - center) @ z_axis
    h_min, h_max = np.min(projections), np.max(projections)
    height   = (h_max - h_min) * 1.1
    z_offset = (h_max + h_min) / 2.0

    # Unit-radius cylinder aligned with local Z
    mesh = pv.Cylinder(
        center=(0, 0, z_offset),
        direction=(0, 0, 1),
        radius=1.0,
        height=height,
        resolution=100,
    )

    # Deform into an ellipse using the semi-axes
    mesh.points[:, 0] *= a
    mesh.points[:, 1] *= b

    # Move from local frame to world frame using the pose matrix T
    mesh.transform(T, inplace=True)

    return mesh

def visualize_saved_cylinder(scan_num, h5_path='HDF5_DATA/7_SCANS.h5'):
    """
    Reads the HDF5 file and visualizes the Point Cloud, Initial Cylinder, 
    and Optimized Cylinder for a specific scan.
    """
    scan_key = f"{scan_num:04d}"
    
    # ── 1. Load Data from HDF5 ─────────────────────────────────────────
    with h5py.File(h5_path, 'r') as f:
        if f"SCANS/{scan_key}" not in f:
            print(f"Error: Scan {scan_key} not found in {h5_path}")
            return
            
        grp = f[f"SCANS/{scan_key}"]
        
        # Load the point cloud
        point_cloud = grp['POINT_CLOUD'][:]
        
        if 'SURFACE_PRIOR' not in grp:
            print(f"Error: SURFACE_PRIOR data not found for Scan {scan_key}")
            return
            
        prior_grp = grp['SURFACE_PRIOR']
        
        # Load Initial parameters (use [()] to extract scalar values from h5py datasets)
        T_init = prior_grp['INITIAL/T'][:]
        a_init = prior_grp['INITIAL/a'][()]
        b_init = prior_grp['INITIAL/b'][()]
        
        # Load Optimized parameters
        T_opt = prior_grp['OPTIMIZED/T'][:]
        a_opt = prior_grp['OPTIMIZED/a'][()]
        b_opt = prior_grp['OPTIMIZED/b'][()]
        rms   = prior_grp['OPTIMIZED/rms_error'][()]

    # ── 2. Build the Meshes ────────────────────────────────────────────
    mesh_init = _make_elliptic_cylinder_mesh(T_init, a_init, b_init, point_cloud)
    mesh_opt  = _make_elliptic_cylinder_mesh(T_opt, a_opt, b_opt, point_cloud)

    # ── 3. Render ──────────────────────────────────────────────────────
    plotter = pv.Plotter()

    # Point cloud
    plotter.add_points(
        point_cloud,
        color='lightblue',
        point_size=4.5,
        render_points_as_spheres=True,
        opacity=0.8,
        label='Sonar points',
    )

    # Initial cylinder — purple
    plotter.add_mesh(
        mesh_init,
        color='purple',
        opacity=0.30,
        show_edges=False,
        label=f'Initial (a={a_init:.2f}, b={b_init:.2f})',
    )

    # Optimized cylinder — orange
    plotter.add_mesh(
        mesh_opt,
        color='orange',
        opacity=0.40,
        show_edges=False,
        label=f'Optimized (a={a_opt:.2f}, b={b_opt:.2f}, RMS={rms:.3f})',
    )

    plotter.add_axes()
    plotter.add_legend(size=(0.35, 0.15), bcolor=(0.95, 0.95, 0.95))
    plotter.add_title(f'Scan {scan_num:04d} — Saved Surface Priors', font_size=12)

    # Note: ensure trame is running if you are in a Jupyter environment
    plotter.show(jupyter_backend='trame')

visualize_saved_cylinder(57)

# 3) $f \leftarrow \mathcal{GP}(\mathbf{s}, \mathcal{M}^v, f_p)$

You're learning a function f(s, ψᵛ) → ρᵛ that maps a position along the trajectory (curvilinear abscissa s) and a vertical sonar rotation angle ψᵛ to a predicted range ρᵛ. The GP doesn't model the raw range directly — it models the residual between the actual measured range and the prior shape fₚ. So the GP learns the deviations from the elliptic cylinder you fitted in line 2.

In [ ]:
from gp_surface import train_gp_single_scan, train_gp_all_scans, save_gp_params

# # --- Test on one scan first ---
gp_result, scan_data = train_gp_single_scan(57, h5_path='HDF5_DATA/7_SCANS.h5', prior_key='INITIAL')
# print(gp_result['model'])

# # --- Train all scans ---
gp_models = train_gp_all_scans(h5_path='HDF5_DATA/7_SCANS.h5', prior_key='INITIAL')

# # --- Save hyperparameters + training data to HDF5 ---
save_gp_params(gp_models, h5_path='HDF5_DATA/7_SCANS.h5')

## visualize


In [ ]:
from pyvista.trame.jupyter import launch_server
await launch_server().ready  # Explicitly starts the 3D rendering server

from viz_gp_surface import visualize_gp_surface

# Train + visualize in one call:
gp_result, surface_data = visualize_gp_surface(
    scan_num          = 292,               # Your scan number
    show_bounds       = False,       # Turns off the +/- 3 sigma translucent shells
    show_cylinder     = False,        # Turns off the orange wireframe prior
    show_training_pts = True      # Turns off the red sonar dots
)
# Or reuse an already-trained GP:
# gp_result, surface_data = visualize_gp_surface(57, gp_result=gp_result)

# contor

In [ ]:
from viz_gp_surface import visualize_gp_contours

gp_result = visualize_gp_contours(57, n_s_slices=4)

# 4) for  

In [5]:
"""
=============================================================================
Algorithm 2 (Breux & Lapierre 2020) — Elevation Angle MLE
LINES 1–11   (preparation, per-θ_q geometry, and segment selection)
=============================================================================

This module implements the first eleven lines of Algorithm 2:

    1:  Θ̂ ← ∅,   Δ_θ ← (θ_max − θ_min) / n_q
    2:  for q do
    3:      θ_q ← θ_{q-1} + Δ_θ
    4:      x ← g^h([ρ_k^h, θ_q, ψ_k^h], k)             ▷ Equation (20)
    5:      I ← ∅
    6:      for i ∈ [1, n_r − 1] do
    7:          n^v_{|{0},i}, n^v_{|{0},i+1}            ▷ Equation (21)
                ← R(q_i)R(q_V) n^v_{|{S^V}},  R(q_{i+1})R(q_V) n^v_{|{S^V}}
    8:          if ⟨n^v_{|{0},i},   x − τ_i⟩
                 · ⟨n^v_{|{0},i+1}, x − τ_{i+1}⟩  < 0  then    ▷ Equation (22)
    9:              I ← I ∪ {i}
   10:          end if
   11:      end for

Lines 12–33 (interpolation, GP query, log-likelihood, Θ̂ collection) can be
appended later without changing what's here.

INPUTS expected (mirrors the paper's notation):

    H_scan          (m, 7) trajectory of this scan  [x, y, z, qw, qx, qy, qz]
    s_scan          (m,)   curvilinear abscissa for each pose in H_scan
    M_h             (3, n_h)  rows = [ρ_h, θ_h (NaN), ψ_h]
    pose_idx_h      (n_h,)    σ^h(k) — *global* pose index for each ping
    pose_idx_min    int       so that local_idx = pose_idx_h - pose_idx_min
    p_H             (7,)      pose of horizontal sonar in robot body frame
                              [x, y, z, qw, qx, qy, qz]   (extrinsic calibration)
    p_V             (7,)      pose of vertical sonar in robot body frame
                              [x, y, z, qw, qx, qy, qz]   (extrinsic calibration)
    n_v_local       (3,)      rotation axis of the vertical sonar in {S^V}
                              (typically [0, 0, 1] — the Z-axis of {S^V})
    beam_width      float     vertical aperture of the horizontal sonar (rad)
    n_q             int       number of θ samples (paper uses 200)
    k               int       which horizontal ping to process (0-indexed)

OUTPUTS (per ping k):

    A list of length n_q.  Element q is a dict with:
        'theta_q' : float          — the candidate elevation angle (rad)
        'x'       : (3,) np.array  — candidate 3D world point
        'I'       : list[int]      — segment-start indices i where x(θ_q) crosses
                                     the vertical sonar plane (filled in lines 6–11)

Notes on g^h (Equation 1 of the paper):
---------------------------------------
The paper defines:
    g^h([ρ, θ, ψ], k) =  η_{σ^h(k)}  ⊕  p^H  ⊕  [ρ cos θ cos ψ,
                                                  ρ cos θ sin ψ,
                                                  ρ sin θ]

Here ⊕ is the SE(3) pose-point composition.  Concretely:
    1) build local point in {S^H}:    p_local = [ρcθcψ, ρcθsψ, ρsθ]
    2) lift to body frame {B}:        p_body  = R(q^H) p_local + τ^H
    3) lift to world frame {0}:       x       = R(q_robot) p_body + τ_robot

⚠️  IMPORTANT — coordinate convention:
    Your existing `compute_point_cloud` (used for the *vertical* sonar) applies
    two extra alignment rotations (Z by π/2, then Y by π/2) before the final
    quaternion rotation.  Those rotations encode how the sonar is mounted on
    the robot.  For the *horizontal* sonar you need the analogous extrinsic
    calibration p^H.  In this module we assume p^H is supplied as a pose
    [x, y, z, qw, qx, qy, qz] in the body frame, with the convention that the
    sonar's XY plane (where ψ rotates and where θ=0 lies) is already encoded
    in p^H.  If your data uses a different convention, adapt `g_h` below or
    bake the alignment into p^H once and for all.
=============================================================================
"""

import numpy as np
from scipy.spatial.transform import Rotation


# ─────────────────────────────────────────────────────────────────────────────
# SE(3) helpers — match the conventions used in Algorithm_1.ipynb
# (quaternion stored as [qw, qx, qy, qz], scalar-first)
# ─────────────────────────────────────────────────────────────────────────────

def _pose_to_R_t(pose):
    """
    Split a 7-vector pose into a 3×3 rotation matrix and a 3-vector translation.

    pose = [x, y, z, qw, qx, qy, qz]   (scalar-first quaternion)
    """
    t = np.asarray(pose[:3], dtype=float)
    qw, qx, qy, qz = pose[3:]
    # scipy uses scalar-LAST: [qx, qy, qz, qw]
    R = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()
    return R, t


def _compose_pose_point(pose, p_local):
    """
    SE(3) pose ⊕ point  composition:    p_out = R(pose) @ p_local + t(pose)

    This is exactly the ⊕ operator used in Equation (1) of the paper for the
    pose-point case.
    """
    R, t = _pose_to_R_t(pose)
    return R @ p_local + t


def _pose_R(pose):
    """Return only the 3×3 rotation matrix from a 7-vector pose."""
    R, _ = _pose_to_R_t(pose)
    return R


def _vertical_axis_world(pose_robot, p_V, n_v_local):
    """
    Equation (21):   n^v_{|{0}, i}  =  R(q_i) · R(q_V) · n^v_{|{S^V}}

    Compute the world-frame direction of the vertical sonar's rotation axis
    when the robot is at the given pose.  Geometrically this is the normal
    vector of the vertical sonar's instantaneous scanning plane.

    Parameters
    ----------
    pose_robot : (7,)  robot pose in world frame    [x, y, z, qw, qx, qy, qz]
    p_V        : (7,)  vertical sonar pose in body  [x, y, z, qw, qx, qy, qz]
    n_v_local  : (3,)  rotation axis in {S^V}       (typically [0, 0, 1])

    Returns
    -------
    n_v_world  : (3,)  rotation axis expressed in {0}, unit-length-preserving
    """
    R_robot = _pose_R(pose_robot)        # body  → world
    R_V     = _pose_R(p_V)               # sonar → body
    return R_robot @ R_V @ np.asarray(n_v_local, dtype=float)


# ─────────────────────────────────────────────────────────────────────────────
# Line 4 — Equation (20):  x ← g^h([ρ_k^h, θ_q, ψ_k^h], k)
# ─────────────────────────────────────────────────────────────────────────────

def g_h(rho, theta, psi, k, H_scan, p_H, pose_idx_h, pose_idx_min):
    """
    Map a horizontal-sonar spherical measurement to a 3D world point.

    Implements Equation (1) / Equation (20):

        x = η_{σ^h(k)}  ⊕  p^H  ⊕  [ρ cos θ cos ψ,
                                     ρ cos θ sin ψ,
                                     ρ sin θ]

    Parameters
    ----------
    rho           : float  — measured range of ping k  (ρ_k^h)
    theta         : float  — HYPOTHESIZED elevation angle θ_q  (rad)
    psi           : float  — measured azimuth of ping k  (ψ_k^h, rad)
    k             : int    — index of the horizontal ping
    H_scan        : (m, 7) — trajectory poses for this scan (local, 0-indexed)
    p_H           : (7,)   — sonar→body extrinsic [x,y,z,qw,qx,qy,qz]
    pose_idx_h    : (n_h,) — *global* pose index per ping (σ^h(k))
    pose_idx_min  : int    — global→local pose offset for this scan

    Returns
    -------
    x : (3,) np.array — candidate 3D world-frame point
    """
    # --- Step 1: Build the spherical-coordinate point in the sonar frame {S^H}
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    cos_p, sin_p = np.cos(psi),   np.sin(psi)
    p_local = rho * np.array([cos_t * cos_p,
                              cos_t * sin_p,
                              sin_t])

    # --- Step 2:  p^H ⊕ p_local      (sonar frame → body frame)
    p_body = _compose_pose_point(p_H, p_local)

    # --- Step 3:  η_{σ^h(k)} ⊕ p_body      (body frame → world frame)
    local_idx = int(pose_idx_h[k]) - int(pose_idx_min)
    local_idx = max(0, min(local_idx, H_scan.shape[0] - 1))   # safety clamp
    eta_k = H_scan[local_idx]
    x = _compose_pose_point(eta_k, p_body)

    return x


# ─────────────────────────────────────────────────────────────────────────────
# Algorithm 2 — Lines 1 through 11 (per single horizontal ping k)
# ─────────────────────────────────────────────────────────────────────────────

def run_lines_1_to_11(M_h, pose_idx_h, pose_idx_min, H_scan,
                      p_H, p_V, k, beam_width, n_q,
                      n_v_local=np.array([0.0, 0.0, 1.0])):
    """
    Execute lines 1–11 of Algorithm 2 for ONE horizontal sonar measurement.

    Algorithm 2 lines:
        1:  Θ̂ ← ∅,   Δ_θ ← (θ_max − θ_min) / n_q
        2:  for q do
        3:      θ_q ← θ_{q-1} + Δ_θ
        4:      x ← g^h([ρ_k^h, θ_q, ψ_k^h], k)        ▷ Equation (20)
        5:      I ← ∅
        6:      for i ∈ [1, n_r − 1] do
        7:          n^v_{|{0},i}, n^v_{|{0},i+1}       ▷ Equation (21)
        8:          if  ⟨n^v_i,   x − τ_i⟩
                      · ⟨n^v_{i+1}, x − τ_{i+1}⟩  < 0  then    ▷ Equation (22)
        9:              I ← I ∪ {i}
       10:          end if
       11:      end for

    Parameters
    ----------
    M_h           : (3, n_h)   horizontal sonar matrix.  Rows: [ρ, θ(NaN), ψ]
    pose_idx_h    : (n_h,)     σ^h — global pose index of each ping
    pose_idx_min  : int        global→local pose offset for this scan
    H_scan        : (m, 7)     trajectory poses during this scan
    p_H           : (7,)       horizontal sonar pose in body frame
    p_V           : (7,)       vertical sonar pose in body frame
    k             : int        which ping to process (0-indexed)
    beam_width    : float      vertical aperture of the h-sonar (rad).
                               θ_max = +b/2,  θ_min = −b/2
    n_q           : int        number of θ samples
    n_v_local     : (3,)       rotation axis of the vertical sonar in {S^V}.
                               Default = [0, 0, 1] (Z-axis of the sonar frame).

    Returns
    -------
    state : dict with the following keys
        'Theta_hat'  : list   — empty Θ̂  (filled later by line 31)
        'Delta_theta': float  — angular step Δ_θ (line 1)
        'theta_min'  : float
        'theta_max'  : float
        'rho_k'      : float  — measured range ρ_k^h
        'psi_k'      : float  — measured azimuth ψ_k^h
        'n_v_world'  : (m, 3) — Eq. (21) precomputed for every pose in H_scan
                                (reused inside lines 6–11; useful downstream)
        'samples'    : list of length n_q.  Each element is a dict:
            {
              'q'      : int,
              'theta_q': float,         # line 3
              'x'      : np.array(3,),  # line 4
              'I'      : list[int],     # lines 5 + 6–11
            }
    """
    # ────────────────────────────────────────────────────────────────────
    # Read the k-th horizontal ping
    # ────────────────────────────────────────────────────────────────────
    rho_k = float(M_h[0, k])     # range
    # M_h[1, k] is θ — currently NaN, this is what we are estimating
    psi_k = float(M_h[2, k])     # azimuth

    # ────────────────────────────────────────────────────────────────────
    # LINE 1:  Θ̂ ← ∅,   Δ_θ ← (θ_max − θ_min) / n_q
    # ────────────────────────────────────────────────────────────────────
    Theta_hat   = []                    # empty set, to be filled later (line 31)
    theta_max   =  beam_width / 2.0     # +b/2
    theta_min   = -beam_width / 2.0     # −b/2
    Delta_theta = (theta_max - theta_min) / n_q

    # ────────────────────────────────────────────────────────────────────
    # PRECOMPUTATION FOR LINE 7 (Equation 21)
    #
    #   n^v_{|{0}, i}  =  R(q_i) · R(q_V) · n^v_{|{S^V}}
    #
    # This depends ONLY on the trajectory (q_i) and the fixed sonar mounting
    # (q_V) — NOT on the candidate elevation angle θ_q.  Computing it once
    # outside the q-loop avoids redoing the same n_r matrix products n_q
    # times inside the loop.
    # ────────────────────────────────────────────────────────────────────
    n_r       = H_scan.shape[0]
    n_v_world = np.zeros((n_r, 3))
    for i in range(n_r):
        n_v_world[i] = _vertical_axis_world(H_scan[i], p_V, n_v_local)

    # Translation parts τ_i of every pose — also constant w.r.t. θ_q
    tau_all = H_scan[:, :3]             # shape (n_r, 3)

    # ────────────────────────────────────────────────────────────────────
    # LINE 2: for q do
    # ────────────────────────────────────────────────────────────────────
    samples = []

    # The recurrence in line 3 is θ_q = θ_{q-1} + Δ_θ.  With Θ̂ initialized
    # at line 1 and the loop starting at q=1, the first value is therefore
    # θ_1 = θ_min + Δ_θ.  We follow that exactly.
    theta_prev = theta_min

    for q in range(1, n_q + 1):
        # ────────────────────────────────────────────────────────────────
        # LINE 3:  θ_q ← θ_{q-1} + Δ_θ
        # ────────────────────────────────────────────────────────────────
        theta_q   = theta_prev + Delta_theta
        theta_prev = theta_q

        # ────────────────────────────────────────────────────────────────
        # LINE 4:  x ← g^h([ρ_k^h, θ_q, ψ_k^h], k)        ▷ Equation (20)
        # ────────────────────────────────────────────────────────────────
        x = g_h(rho=rho_k, theta=theta_q, psi=psi_k, k=k,
                H_scan=H_scan, p_H=p_H,
                pose_idx_h=pose_idx_h, pose_idx_min=pose_idx_min)

        # ────────────────────────────────────────────────────────────────
        # LINE 5:  I ← ∅
        #   I will collect indices i of trajectory segments [i, i+1] where
        #   the candidate point x(θ_q) crosses the vertical sonar plane.
        # ────────────────────────────────────────────────────────────────
        I = []

        # ────────────────────────────────────────────────────────────────
        # LINES 6–11:   for i ∈ [1, n_r − 1] ...   end for
        #
        #   For each consecutive pose pair, test the intermediate value
        #   theorem condition (Eq. 22):
        #
        #       ⟨n^v_{|{0},i},   x − τ_i⟩  ·  ⟨n^v_{|{0},i+1}, x − τ_{i+1}⟩  < 0
        #
        #   If true, the signed distance from x to the vertical sonar plane
        #   flips sign between pose i and pose i+1, so the plane crosses x
        #   somewhere inside that segment.  That segment becomes a candidate
        #   for the root-finding step (line 18).
        #
        #   Vectorized: compute all n_r dot products at once, then check
        #   sign changes between consecutive entries.
        # ────────────────────────────────────────────────────────────────
        # Signed perpendicular distances of x to each pose's sonar plane
        d = np.einsum('ij,ij->i', n_v_world, x - tau_all)   # shape (n_r,)

        # Sign-change test on consecutive pairs (line 8)
        sign_change = d[:-1] * d[1:] < 0                    # shape (n_r-1,)

        # Line 9:  I ← I ∪ {i}   for every i where the test fires
        # (Indices are 0-based here; the paper uses 1-based but the meaning
        #  is identical — i refers to the *starting* pose of segment [i, i+1].)
        I = np.where(sign_change)[0].tolist()

        samples.append({
            'q'       : q,
            'theta_q' : theta_q,
            'x'       : x,
            'I'       : I,
        })

    return {
        'Theta_hat'   : Theta_hat,
        'Delta_theta' : Delta_theta,
        'theta_min'   : theta_min,
        'theta_max'   : theta_max,
        'rho_k'       : rho_k,
        'psi_k'       : psi_k,
        'n_v_world'   : n_v_world,    # (n_r, 3)  — Eq. (21) per pose
        'samples'     : samples,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Convenience wrapper — load a scan from HDF5 and run lines 1–11 for one ping
# ─────────────────────────────────────────────────────────────────────────────

def run_lines_1_to_11_from_h5(scan_num, k, p_H, p_V,
                              h5_path='HDF5_DATA/7_SCANS.h5',
                              beam_width=np.radians(35.0),
                              n_q=200,
                              n_v_local=np.array([0.0, 0.0, 1.0])):
    """
    Convenience wrapper: pull a scan straight from the HDF5 file and run
    lines 1–11 for the k-th horizontal ping.

    Parameters
    ----------
    scan_num   : int    — scan number (e.g. 57)
    k          : int    — index of the horizontal ping
    p_H        : (7,)   — horizontal-sonar extrinsic in body frame
                          [x, y, z, qw, qx, qy, qz]
    p_V        : (7,)   — vertical-sonar extrinsic in body frame
                          [x, y, z, qw, qx, qy, qz]
    h5_path    : str
    beam_width : float  — radians (default 35°)
    n_q        : int
    n_v_local  : (3,)   — vertical sonar rotation axis in {S^V}, default [0,0,1]

    Returns
    -------
    state      : dict (same as run_lines_1_to_11)
    scan_data  : dict with the raw scan arrays (handy for downstream lines)
    """
    import h5py

    sk = f"{scan_num:04d}"
    with h5py.File(h5_path, 'r') as f:
        scan = f[f'SCANS/{sk}']

        H_scan       = scan['H'][:]
        s_scan       = scan['s'][:]
        pose_idx_min = int(scan['pose_idx_min'][()])

        M_h          = scan['M_h_M'][:]                    # shape (3, n_h)
        pose_idx_h   = scan['M_h_pose_idx'][:].astype(int) # shape (n_h,)

    scan_data = {
        'H_scan'      : H_scan,
        's_scan'      : s_scan,
        'pose_idx_min': pose_idx_min,
        'M_h'         : M_h,
        'pose_idx_h'  : pose_idx_h,
    }

    state = run_lines_1_to_11(
        M_h=M_h, pose_idx_h=pose_idx_h, pose_idx_min=pose_idx_min,
        H_scan=H_scan, p_H=p_H, p_V=p_V, k=k,
        beam_width=beam_width, n_q=n_q,
        n_v_local=n_v_local,
    )

    return state, scan_data


# ─────────────────────────────────────────────────────────────────────────────
# Quick self-test
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    # ─────────────────────────────────────────────────────────────────────
    # Synthetic scenario for lines 1–11.
    #
    # Setup:
    #   • Robot moves along the world X-axis from x=-2 to x=+2 in 5 steps.
    #   • Identity orientation at every pose.
    #   • Identity sonar extrinsics (p_H = p_V = identity).
    #   • Vertical sonar rotation axis n^v_{|S^V} = [0, 0, 1] (Z of {S^V}).
    #     With identity orientation this stays [0, 0, 1] in world frame,
    #     so the vertical sonar plane at each pose is the XY plane through
    #     that pose, with the Z component giving the signed distance.
    #   • Horizontal ping: ρ=2, ψ=0, recorded at the middle pose (x=0).
    #     With θ_q = 0, x ≈ [2, 0, 0] — Z=0, lies *exactly* on every
    #     sonar plane → no sign change → I should be empty.
    #   • With θ_q > 0, x has Z>0 — same side everywhere → I empty.
    #
    # To force a sign change we shift the trajectory's Z so that some
    # poses are above x and some below.  Below we tilt the trajectory in
    # Z so the IVT condition triggers.
    # ─────────────────────────────────────────────────────────────────────
    H_scan = np.array([
        [-2.0, 0.0, -1.0, 1, 0, 0, 0],   # pose 0  (below x in Z)
        [-1.0, 0.0, -0.5, 1, 0, 0, 0],
        [ 0.0, 0.0,  0.0, 1, 0, 0, 0],
        [ 1.0, 0.0,  0.5, 1, 0, 0, 0],
        [ 2.0, 0.0,  1.0, 1, 0, 0, 0],   # pose 4  (above x in Z)
    ])
    p_H        = np.array([0, 0, 0, 1, 0, 0, 0])
    p_V        = np.array([0, 0, 0, 1, 0, 0, 0])
    pose_idx_h = np.array([2])                       # ping fired at pose 2
    M_h        = np.array([[2.0],     # ρ
                           [np.nan],  # θ unknown
                           [0.0]])    # ψ

    state = run_lines_1_to_11(
        M_h=M_h, pose_idx_h=pose_idx_h, pose_idx_min=0,
        H_scan=H_scan, p_H=p_H, p_V=p_V, k=0,
        beam_width=np.radians(35.0), n_q=5,
    )

    print(f"Δ_θ          = {state['Delta_theta']:.4f}  rad")
    print(f"# trajectory poses (n_r)    = {H_scan.shape[0]}")
    print(f"# segments to test          = {H_scan.shape[0] - 1}")
    print(f"n_v_world (should be ~Z at every pose):")
    print(state['n_v_world'])
    print()
    for s in state['samples']:
        print(f"  q={s['q']}  θ_q={s['theta_q']:+.4f}  "
              f"x={s['x']}  I={s['I']}")

    # Sanity check: with θ_q ≈ 0, x ≈ [2, 0, 0] and the trajectory's Z
    # ranges from -1 to +1, so the signed distance to each sonar plane
    # (here Z-coordinate of x − τ_i, since the axis is +Z) goes from
    # +1 down through 0 to -1 along the trajectory.  We therefore expect
    # the middle θ samples to produce non-empty I.

Δ_θ          = 0.1222  rad
# trajectory poses (n_r)    = 5
# segments to test          = 4
n_v_world (should be ~Z at every pose):
[[0. 0. 1.]
 [0. 0. 1.]
 [0. 0. 1.]
 [0. 0. 1.]
 [0. 0. 1.]]

  q=1  θ_q=-0.1833  x=[ 1.96650982  0.         -0.36447105]  I=[1]
  q=2  θ_q=-0.0611  x=[ 1.9962696   0.         -0.12209708]  I=[1]
  q=3  θ_q=+0.0611  x=[1.9962696  0.         0.12209708]  I=[2]
  q=4  θ_q=+0.1833  x=[1.96650982 0.         0.36447105]  I=[2]
  q=5  θ_q=+0.3054  x=[1.9074339 0.        0.6014116]  I=[3]
